In [1]:
import tqdm
import pickle
import warnings
import numpy as np
import pandas as pd
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from data import *
from model import *
from utils import *
from recourse_model import ROAR, LAROAR, RecourseCost, RobustRecourse

warnings.filterwarnings('ignore')

In [2]:
def append_res(d, rob, loss, cost, m1_validity, wc_validity):
    d['J'].append(rob)
    d['Loss'].append(loss)
    d['Cost'].append(cost)
    d['M1 Validity'].append(m1_validity)
    d['WC Validity'].append(wc_validity)
    
def get_res(d):
    result = {
        'J': np.mean(d['J']),
        'Loss': np.mean(d['Loss']),
        'Cost': np.mean(d['Cost']),
        'M1 Validity': np.mean(d['M1 Validity']),
        'WC Validity': np.mean(d['WC Validity']),
    }
    return result

In [3]:
def recourse_model_runner(X: np.ndarray, recourse_model1: LAROAR, recourse_model2: ROAR, base_model: Model, X_train):    
    model_adv_opt = deepcopy(base_model)
    model_adv_roar = deepcopy(base_model)
    
    results_opt = {'J': [], 'Loss': [], 'Cost': [], 'M1 Validity': [], 'WC Validity': []}
    results_roar = deepcopy(results_opt)
    
    n = len(X)
    for i in tqdm.trange(n, desc=f'Eval alpha={recourse_model1.alpha}; lambda={recourse_model1.lamb}', colour='#0091ff'):
        x_0 = X[i]
        J = RecourseCost(x_0, recourse_model1.lamb)
        
        if isinstance(base_model, NN):
            #set seed for lime
            np.random.seed(i)
            weights, bias = lime_explanation(base_model.predict, 
                X_train, x_0)
            weights, bias = np.round(weights, 4), np.round(bias, 4)
            
            recourse_model1.weights = weights
            recourse_model1.bias = bias
            
            recourse_model2.set_weights(weights)
            recourse_model2.set_bias(bias)
        
        # OPT
        x_r = recourse_model1.get_recourse(x_0, beta=1.)
        weights_r, bias_r = recourse_model1.calc_theta_adv(x_r)
        bce_loss_opt, cost_opt, opt_rob = J(x_r, weights_r, bias_r, True)
        # m1_validity_opt = base_model.predict_proba(x_r.reshape(1,-1))[0,1]
        m1_validity_opt = base_model.predict(x_r.reshape(1,-1))[0]
        
        model_adv_opt.model.coef_ = weights_r.reshape(1,-1)
        model_adv_opt.model.intercept_ = bias_r
        # wc_validity_opt = model_adv_opt.predict_proba(x_r.reshape(1,-1))[0,1]
        wc_validity_opt = model_adv_opt.predict(x_r.reshape(1,-1))[0]
        
        # ROAR
        x_r2, _ = recourse_model2.get_recourse(x_0)
        weights_r2, bias_r2 = recourse_model1.calc_theta_adv(x_r2)
        bce_loss_roar, cost_roar, rob_roar = J(x_r2, weights_r2, bias_r2, True)
        # m1_validity_roar = base_model.predict_proba(x_r2.reshape(1,-1))[0,1]
        m1_validity_roar = base_model.predict(x_r2.reshape(1,-1))[0]
        
        model_adv_roar.model.coef_ = weights_r.reshape(1,-1)
        model_adv_roar.model.intercept_ = bias_r
        # wc_validity_roar = model_adv_roar.predict_proba(x_r2.reshape(1,-1))[0,1]
        wc_validity_roar = model_adv_roar.predict(x_r2.reshape(1,-1))[0]
        
        append_res(results_opt, opt_rob, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt)
        append_res(results_roar, rob_roar, bce_loss_roar, cost_roar, m1_validity_roar, wc_validity_roar)
    
    return get_res(results_opt), get_res(results_roar)

In [4]:
def append_local_res(d, res):
    d['J'].append(res['J'])
    d['Loss'].append(res['Loss'])
    d['Cost'].append(res['Cost'])
    d['M1 Validity'].append(res['M1 Validity'])
    d['WC Validity'].append(res['WC Validity'])
    
def get_final_res(d, id, alpha, lamb):
    results = {
        'id': [id],
        'alpha': [alpha],
        'lambda': [lamb],
        'Mean Cost': [np.mean(d['Cost'])],
        'Std Cost': [np.std(d['Cost'], ddof=1)],
        'Mean M1 Validity': [np.mean(d['M1 Validity'])],
        'Std M1 Validity': [np.std(d['M1 Validity'], ddof=1)],
        'Mean WC Validity': [np.mean(d['WC Validity'])],
        'Std WC Validity': [np.std(d['WC Validity'], ddof=1)],
        'Mean J': [np.mean(d['J'])],
        'Std J': [np.std(d['J'], ddof=1)],
        'Mean Loss': [np.mean(d['Loss'])],
        'Std Loss': [np.std(d['Loss'], ddof=1)],
    }
    return pd.DataFrame(results)

In [5]:
def recourse_tradeoff(params: dict, hyper_params: dict, seeds: list, results: dict):
    if params['data'] in ['correction', 'german']:
        data_model = CorrectionShift("../datasets/german.csv", "../datasets/corrected_german.csv", seed=0)
    elif params['data'] in ['temporal', 'business']:
        data_model = TemporalShift("../datasets/SBAcase.11.13.17.csv", seed=0)
    elif params['data'] in ['geospatial', 'student']:
        data_model = GeospatialShift("../datasets/student-por.csv", seed=0)
    elif params['data'] in ['synthetic', 'simulated']:
        data_model = SyntheticData(alpha=1.5, beta=0, n=1000, seed=0)
    
    for alpha in hyper_params['alpha']:
        for lamb in hyper_params['lambda']:
            local_res_opt = {'J': [], 'Loss': [], 'Cost': [], 'WC Validity': [], 'M1 Validity': []}
            local_res_roar = {'J': [], 'Loss': [], 'Cost': [], 'WC Validity': [], 'M1 Validity': []}
            for seed in seeds:
                data1, data2 = data_model.get_data(seed)
                X1_train, y1_train, X1_test, y1_test = data1

                if params['base_model'] == 'lr':
                    base_model = LR()
                elif params['base_model'] == 'nn':
                    base_model = NN(X1_train.shape[1])
                    
                base_model.train(X1_train.values, y1_train.values)
                
                recourse_needed_X1_test = recourse_needed(base_model.predict, X1_test.values)
                
                weights, bias = None, None
                if params['base_model'] == 'lr':
                    weights = base_model.model.coef_[0]
                    bias = base_model.model.intercept_
                
                recourse_model1 = LAROAR(
                    weights = weights,
                    bias = bias,
                    alpha = alpha,
                    lamb = lamb
                )    
                recourse_model2 = ROAR(
                    weights = weights,
                    bias = bias,
                    alpha = alpha,
                    lamb = lamb
                )
                
                result_opt, result_roar = recourse_model_runner(recourse_needed_X1_test, recourse_model1, recourse_model2, base_model, X1_train)
                append_local_res(local_res_opt, result_opt)
                append_local_res(local_res_roar, result_roar)
                
            results['tradeoffs'] = pd.concat((results['tradeoffs'], get_final_res(local_res_opt, 'OPT', alpha, lamb), get_final_res(local_res_roar, 'ROAR', alpha, lamb)))
            print()
    return results

In [6]:
torch.manual_seed(0)
params = {}
# 'synthetic/simulated', 'correction/german', 'temporal/business', 'geospatial/student'
params['data'] = 'synthetic'
# 'lr', 'nn
params['base_model'] = 'lr'

hyper_params = {}
hyper_params['lambda'] = np.linspace(0.1, 1.0, 10).round(1)
hyper_params['alpha'] = np.linspace(0.1, 1.0, 10).round(1)
seeds = list(range(5))

results = {'tradeoffs': pd.DataFrame()}
recourse_tradeoff(params, hyper_params, seeds, results)

Eval alpha=0.1; lambda=0.1: 100%|██████████| 93/93 [02:04<00:00,  1.34s/it]


Eval alpha=0.1; lambda=0.2: 100%|██████████| 93/93 [01:53<00:00,  1.22s/it]


Eval alpha=0.1; lambda=0.3: 100%|██████████| 93/93 [01:46<00:00,  1.15s/it]


Eval alpha=0.1; lambda=0.4: 100%|██████████| 93/93 [01:42<00:00,  1.10s/it]


Eval alpha=0.1; lambda=0.5: 100%|██████████| 93/93 [01:39<00:00,  1.07s/it]


...

Eval alpha=1.0; lambda=0.7: 100%|██████████| 93/93 [01:34<00:00,  1.01s/it]


Eval alpha=1.0; lambda=0.8: 100%|██████████| 93/93 [01:41<00:00,  1.09s/it]


Eval alpha=1.0; lambda=0.9: 100%|██████████| 93/93 [01:37<00:00,  1.05s/it]


Eval alpha=1.0; lambda=1.0: 100%|██████████| 93/93 [01:41<00:00,  1.10s/it]

{'tradeoffs':       id  alpha  lambda  Mean Cost  Std Cost  Mean M1 Validity  \
 0    OPT    0.1     0.1   5.610546  0.061912               1.0   
 0   ROAR    0.1     0.1   4.958982  0.057974               1.0   
 0    OPT    0.1     0.2   5.211506  0.059667               1.0   
 0   ROAR    0.1     0.2   4.775550  0.057414               1.0   
 0    OPT    0.1     0.3   4.963460  0.058266               1.0   
 ..   ...    ...     ...        ...       ...               ...   
 0   ROAR    1.0     0.8   4.098588  0.042748               1.0   
 0    OPT    1.0     0.9   4.066043  0.045333               1.0   
 0   ROAR    1.0     0.9   4.068659  0.044427               1.0   
 0    OPT    1.0     1.0   4.066043  0.045333               1.0   
 0   ROAR    1.0     1.0   4.068555  0.045362               1.0   
 
     Std M1 Validity  Mean WC Validity  Std WC Validity    Mean J     Std J  \
 0               0.0               1.0              0.0  0.615660  0.006502   
 0               0.0   

In [7]:
with open(f'../results/params_effect_{params["base_model"]}_{params["data"]}.pkl', 'wb') as f:
    pickle.dump(results['tradeoffs'], f)